# ReefScan-ID: Stage 2 patch dataset

Patches for stage 2 of the cascade: points inside coral, labelled healthy / bleached / dead. Run in Google Colab; this notebook needs no GPU.

### Setup: Colab runtime and Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/ReefScan-ID"
raw_dir = base_dir + "/RawData"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))
print("Cache folder exists: {}".format(os.path.isdir(raw_dir)))
print("CPU count: {}".format(os.cpu_count()))

### Setup: libraries

In [ ]:
!pip install -q datasets

import datasets
import numpy as np
import pandas as pd

print("datasets version: {}".format(datasets.__version__))
print("numpy version: {}".format(np.__version__))
print("pandas version: {}".format(pd.__version__))

## Section 1: Stage 2 label map

Question: which Coralscapes classes become healthy, bleached and dead for stage 2, and how many pixels does each of the three hold in the train and validation splits?

Checks: map every mask value to a stage 2 state, print the classes in each state, count the pixels of every mask value in train and validation, then print the share of each state. The test split is left out here because it is kept for the final evaluation.

### Load the dataset

In [ ]:
import time
from datasets import load_dataset

start = time.time()
ds = load_dataset("EPFL-ECEO/coralscapes", cache_dir=raw_dir)
print(ds)
print("Loading took {:.1f} seconds".format(time.time() - start))

### Class names from the dataset card

Ids and names copied from the table on the Coralscapes dataset card (read 2026-11-12), as in notebook 02.

In [ ]:
class_names = {
    1: "seagrass",
    2: "trash",
    3: "other coral dead",
    4: "other coral bleached",
    5: "sand",
    6: "other coral alive",
    7: "human",
    8: "transect tools",
    9: "fish",
    10: "algae covered substrate",
    11: "other animal",
    12: "unknown hard substrate",
    13: "background",
    14: "dark",
    15: "transect line",
    16: "massive/meandering bleached",
    17: "massive/meandering alive",
    18: "rubble",
    19: "branching bleached",
    20: "branching dead",
    21: "millepora",
    22: "branching alive",
    23: "massive/meandering dead",
    24: "clam",
    25: "acropora alive",
    26: "sea cucumber",
    27: "turbinaria",
    28: "table acropora alive",
    29: "sponge",
    30: "anemone",
    31: "pocillopora alive",
    32: "table acropora dead",
    33: "meandering bleached",
    34: "stylophora alive",
    35: "sea urchin",
    36: "meandering alive",
    37: "meandering dead",
    38: "crown of thorn",
    39: "dead clam",
}

print("Classes on the card: {}".format(len(class_names)))

### Stage 2 label map

In [ ]:
healthy_ids = [6, 17, 22, 25, 28, 31, 34, 36]
bleached_ids = [4, 16, 19, 33]
dead_ids = [3, 20, 23, 32, 37]
excluded_ids = [21, 27]

state_names = {0: "healthy", 1: "bleached", 2: "dead"}
excluded_state = 253
not_coral_state = 255

# Values that are not coral stay at the not coral code
state_map = np.full(256, not_coral_state, dtype=np.uint8)
state_map[healthy_ids] = 0
state_map[bleached_ids] = 1
state_map[dead_ids] = 2
state_map[excluded_ids] = excluded_state

for state_id, state_name in state_names.items():
    names = [class_names[v] for v in class_names if state_map[v] == state_id]
    print("{} ({} ids): {}".format(state_name, len(names), names))
excluded = [class_names[v] for v in class_names if state_map[v] == excluded_state]
print("excluded coral ({} ids): {}".format(len(excluded), excluded))
print("Ids that are not coral: {}".format(sum(1 for v in class_names if state_map[v] == not_coral_state)))

### Count pixels per mask value

In [ ]:
pixel_counts = {}
for split in ["train", "validation"]:
    counts = np.zeros(256, dtype=np.int64)
    masks = ds[split].select_columns(["label"])
    for row in masks:
        mask = np.array(row["label"])
        counts += np.bincount(mask.ravel(), minlength=256)
    pixel_counts[split] = counts
    print("{}: {} masks counted, {} pixels".format(split, masks.num_rows, counts.sum()))

### Pixel share per stage 2 state

In [ ]:
for split in ["train", "validation"]:
    counts = pixel_counts[split]
    total = counts.sum()
    state_pixels = {state_id: counts[state_map == state_id].sum() for state_id in state_names}
    labelled = sum(state_pixels.values())
    for state_id, state_name in state_names.items():
        print("{} {}: {:.2f}% of all pixels, {:.2f}% of the stage 2 coral pixels".format(
            split, state_name, 100 * state_pixels[state_id] / total, 100 * state_pixels[state_id] / labelled))
    excluded_pixels = counts[state_map == excluded_state].sum()
    print("{} excluded coral: {:.2f}% of all pixels".format(split, 100 * excluded_pixels / total))

**Observation**: